# Download, filter, and visualize extracted residual-stream activations

This notebook works with the chunk files produced by `scripts/extract_residual_stream_positions_from_gcs.py`. It downloads the completions index and extracted chunks, selects samples using prompt metadata, loads one residual-stream layer and cached token position, projects the activations with PCA, and visualizes the result.

## 1. Setup

For a fresh Colab runtime, uncomment the clone and authentication commands. Local runs can skip them.

In [9]:
# !git clone -b dev https://github.com/justinshenk/temporal-manifolds.git
%cd temporal-manifolds
# !gcloud auth application-default login
# !mv -n .env.example .env

[Errno 2] No such file or directory: 'temporal-manifolds'
/content/temporal-manifolds


In [10]:
import gc
import json
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from google.cloud import storage
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
load_dotenv(repo_root / '.env')

True

## 2. Configure the workflow

`METADATA_FILTERS` accepts any prompt-metadata field, using dotted paths for nested fields. A scalar requires an exact match; a list keeps records matching any value in that list. Different fields are combined with AND. Use `None` or an empty dictionary to disable filtering. `AGG_BY` lists the flattened metadata fields that define averaging groups; rows with equal values for every listed field are averaged before PCA. The derived `time_horizon_months` field is available here and combines equivalent durations expressed in different units. Use `None` or an empty list to keep individual rows. Set `LAYER_NAME` to `None` to use the final available residual-stream layer. `POSITION_INDEX` refers to the cached indices written by the extractor (0, 1, or 2), not an absolute token index. Set `PCA_MODEL_LOAD_PATH` to transform with an existing fitted PCA model instead of fitting on the selected data, and set `PCA_MODEL_SAVE_PATH` to save the model used by this run. Only load model files from trusted sources.

In [11]:
METADATA_FILTERS: dict[str, object] | None = {
    'template_metadata.prompt_framing': 'task_available_time',
    # 'task_metadata.domain': 'communication',
    'template_metadata.output_format': "approach_actions",
    # 'task': 'answer a yes-or-no question',
}

ACTIVATIONS_GCS_URI = 'gs://temporal-research-bucket/resid_only_0_2'
COMPLETIONS_GCS_URI = 'gs://temporal-research-bucket/completions/completions_256.jsonl'
DATA_DIR = repo_root / 'data' / 'filtered_residual_stream_activations'
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
OVERWRITE = False
MAX_SAMPLES: int | None = None
# None reads every downloaded chunk. Otherwise list local filenames, for example:
# SELECTED_CHUNK_FILES = ['residual_stream_positions_0_2_chunk_00000.pt']
SELECTED_CHUNK_FILES: list[str] | None = None

LAYER_NAME: str | None = "layer_out/21"
POSITION_INDEX = 1
AGG_BY: list[str] | None = ['time_horizon_months']
PCA_MODEL_LOAD_PATH: Path | None = None
PCA_MODEL_SAVE_PATH: Path | None = None  # For example: DATA_DIR / 'pca_model.pkl'
PHRASING_FIELDS = ['template_metadata.prompt_framing', 'template_metadata.output_format']

## 3. Download the extracted chunks and completions index

All extracted `.pt` chunks are cached locally and skipped on subsequent runs unless `OVERWRITE` is true. File selection happens only after the complete download, so changing `SELECTED_CHUNK_FILES` never requires another GCS listing or download.

In [12]:
def parse_gcs_uri(uri):
    if not uri.startswith('gs://'):
        raise ValueError(f'Expected a gs:// URI, got {uri!r}')
    bucket, separator, object_name = uri[5:].partition('/')
    if not bucket or not separator or not object_name.strip('/'):
        raise ValueError(f'GCS URI must contain a bucket and object/prefix: {uri!r}')
    return bucket, object_name.strip('/')


client = storage.Client(project=PROJECT_ID)
DATA_DIR.mkdir(parents=True, exist_ok=True)

completions_bucket, completions_object = parse_gcs_uri(COMPLETIONS_GCS_URI)
COMPLETIONS_PATH = DATA_DIR / Path(completions_object).name
if OVERWRITE or not COMPLETIONS_PATH.exists():
    client.bucket(completions_bucket).blob(completions_object).download_to_filename(str(COMPLETIONS_PATH))

activation_bucket, activation_prefix = parse_gcs_uri(ACTIVATIONS_GCS_URI)
blobs = sorted(
    (blob for blob in client.bucket(activation_bucket).list_blobs(prefix=activation_prefix.rstrip('/') + '/') if blob.name.endswith('.pt')),
    key=lambda blob: blob.name,
)
if not blobs:
    raise FileNotFoundError(f'No .pt chunks found below {ACTIVATIONS_GCS_URI}')

chunk_dir = DATA_DIR / 'chunks'
chunk_dir.mkdir(exist_ok=True)
downloaded_chunk_paths = []
for blob in tqdm(blobs, desc='Downloading chunks'):
    destination = chunk_dir / Path(blob.name).name
    if OVERWRITE or not destination.exists():
        blob.download_to_filename(str(destination))
    downloaded_chunk_paths.append(destination)

print(f'Completions: {COMPLETIONS_PATH}')
print(f'Downloaded/local chunks: {len(downloaded_chunk_paths):,}')

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Completions: /content/temporal-manifolds/data/filtered_residual_stream_activations/completions_256.jsonl
Downloaded/local chunks: 23


### 3.1 Select local chunk files to read

Set `SELECTED_CHUNK_FILES` in the configuration cell to a list of basenames, or leave it as `None` to read every downloaded chunk. This selection does not delete or move any downloaded file.

In [13]:
downloaded_by_name = {path.name: path for path in downloaded_chunk_paths}
if SELECTED_CHUNK_FILES is None:
    selected_chunk_paths = downloaded_chunk_paths
else:
    missing_files = sorted(set(SELECTED_CHUNK_FILES) - set(downloaded_by_name))
    if missing_files:
        raise FileNotFoundError(f'Selected chunk files were not downloaded: {missing_files}')
    selected_chunk_paths = [downloaded_by_name[name] for name in SELECTED_CHUNK_FILES]

if not selected_chunk_paths:
    raise ValueError('No local chunk files were selected.')
print(f'Selected {len(selected_chunk_paths):,} of {len(downloaded_chunk_paths):,} local chunks to read.')
for path in selected_chunk_paths[:10]:
    print(path)

Selected 23 of 23 local chunks to read.
/content/temporal-manifolds/data/filtered_residual_stream_activations/chunks/residual_stream_positions_0_2_chunk_00000.pt
/content/temporal-manifolds/data/filtered_residual_stream_activations/chunks/residual_stream_positions_0_2_chunk_00001.pt
/content/temporal-manifolds/data/filtered_residual_stream_activations/chunks/residual_stream_positions_0_2_chunk_00002.pt
/content/temporal-manifolds/data/filtered_residual_stream_activations/chunks/residual_stream_positions_0_2_chunk_00003.pt
/content/temporal-manifolds/data/filtered_residual_stream_activations/chunks/residual_stream_positions_0_2_chunk_00004.pt
/content/temporal-manifolds/data/filtered_residual_stream_activations/chunks/residual_stream_positions_0_2_chunk_00005.pt
/content/temporal-manifolds/data/filtered_residual_stream_activations/chunks/residual_stream_positions_0_2_chunk_00006.pt
/content/temporal-manifolds/data/filtered_residual_stream_activations/chunks/residual_stream_positions_0_2

## 4. Filter completion records

The JSONL line number is the sample index stored in each extracted chunk. Filters may target top-level or nested prompt-metadata fields (for example, `task`, `base_value`, or `template_metadata.output_format`) and may specify either one value or a list of values to keep. Retain both the selected index set and prompt metadata so activation rows can be joined without relying on chunk order.

In [14]:
_MISSING = object()


def get_metadata_field(metadata, dotted_path):
    value = metadata
    for key in dotted_path.split('.'):
        if not isinstance(value, dict) or key not in value:
            return _MISSING
        value = value[key]
    return value


def metadata_value_matches(actual_value, expected_value):
    if actual_value is _MISSING:
        return False
    allowed_values = expected_value if isinstance(expected_value, list) else [expected_value]
    return any(actual_value == allowed_value for allowed_value in allowed_values)


selected_metadata = {}
with COMPLETIONS_PATH.open(encoding='utf-8') as completion_file:
    for sample_index, line in enumerate(completion_file):
        record = json.loads(line)
        metadata = record['prompt_metadata']
        metadata_matches = all(
            metadata_value_matches(get_metadata_field(metadata, field), expected_value)
            for field, expected_value in (METADATA_FILTERS or {}).items()
        )
        if metadata_matches:
            selected_metadata[sample_index] = metadata
            if MAX_SAMPLES is not None and len(selected_metadata) >= MAX_SAMPLES:
                break

if not selected_metadata:
    raise ValueError('No completion records matched the configured filters.')
selected_indices = set(selected_metadata)
print(f'Selected {len(selected_indices):,} completion records.')
print('First sample indices:', sorted(selected_indices)[:10])

Selected 4,136 completion records.
First sample indices: [8272, 8273, 8274, 8275, 8276, 8277, 8278, 8279, 8280, 8281]


## 5. Load one layer and cached position

The local `.pt` chunks are memory-mapped, so PyTorch does not read every residual layer merely to deserialize the file. A lightweight first pass reads sample-index metadata and determines the exact output size; the second pass touches only the selected layer and position. This relies on the default ZIP serialization used by `torch.save` in the extraction script.

In [15]:
def layer_sort_key(name):
    prefix, separator, suffix = name.rpartition('/')
    return (prefix, int(suffix)) if separator and suffix.isdigit() else (name, name)


def load_chunk_mmap(path):
    return torch.load(path, map_location='cpu', weights_only=True, mmap=True)


# Release large objects left behind if an earlier run was interrupted.
for variable_name in ('payload', 'tensor', 'selected_features', 'activation_matrix', 'feature_parts'):
    globals().pop(variable_name, None)
gc.collect()


preview = load_chunk_mmap(selected_chunk_paths[0])
residuals = preview.get('residual_stream_activations')
if not isinstance(residuals, dict) or not residuals:
    raise ValueError(f'{selected_chunk_paths[0]} has no residual_stream_activations mapping.')
available_layers = sorted(residuals, key=layer_sort_key)
selected_layer = LAYER_NAME or available_layers[-1]
if selected_layer not in residuals:
    raise ValueError(f'Layer {selected_layer!r} is unavailable. Choose from {available_layers}.')
example_tensor = residuals[selected_layer]
if example_tensor.ndim != 3:
    raise ValueError(f'{selected_layer!r} must have shape samples x positions x hidden size.')
hidden_size = example_tensor.shape[-1]
del example_tensor, residuals, preview
gc.collect()
print('Available layers:', available_layers)
print(f'Loading layer={selected_layer!r}, cached position={POSITION_INDEX}.')

read_jobs = []
for path in tqdm(selected_chunk_paths, desc='Scanning chunk metadata'):
    payload = load_chunk_mmap(path)
    chunk_indices = [int(index) for index in payload['sample_indices']]
    row_offsets = [offset for offset, index in enumerate(chunk_indices) if index in selected_indices]
    if row_offsets:
        read_jobs.append((path, row_offsets, [chunk_indices[offset] for offset in row_offsets]))
    del payload

selected_row_count = sum(len(row_offsets) for _path, row_offsets, _indices in read_jobs)
if selected_row_count == 0:
    raise ValueError('None of the selected completion indices occur in the selected chunk files.')
activation_matrix = torch.empty((selected_row_count, hidden_size), dtype=torch.float32)
write_offset = 0
loaded_indices = []
absolute_positions = []
for path, row_offsets, job_indices in tqdm(read_jobs, desc='Reading selected layer'):
    payload = load_chunk_mmap(path)
    cached_indices = list(payload['cached_position_indices'])
    if POSITION_INDEX not in cached_indices:
        raise ValueError(f'{path} does not contain cached position {POSITION_INDEX}.')
    position_offset = cached_indices.index(POSITION_INDEX)
    tensor = payload['residual_stream_activations'][selected_layer]
    rows = torch.as_tensor(row_offsets, dtype=torch.long)
    selected_features = tensor[:, position_offset, :].index_select(0, rows).to(torch.float32)
    next_offset = write_offset + len(row_offsets)
    activation_matrix[write_offset:next_offset].copy_(selected_features)
    write_offset = next_offset
    loaded_indices.extend(job_indices)
    chunk_absolute_positions = payload['absolute_token_positions']
    absolute_positions.extend(chunk_absolute_positions[offset][position_offset] for offset in row_offsets)
    del selected_features, rows, tensor, payload

print('Activation matrix shape:', tuple(activation_matrix.shape))
print(f'Loaded {len(loaded_indices):,} of {len(selected_indices):,} selected samples.')

Available layers: ['layer_out/0', 'layer_out/1', 'layer_out/2', 'layer_out/3', 'layer_out/4', 'layer_out/5', 'layer_out/6', 'layer_out/7', 'layer_out/8', 'layer_out/9', 'layer_out/10', 'layer_out/11', 'layer_out/12', 'layer_out/13', 'layer_out/14', 'layer_out/15', 'layer_out/16', 'layer_out/17', 'layer_out/18', 'layer_out/19', 'layer_out/20', 'layer_out/21', 'layer_out/22', 'layer_out/23', 'layer_out/24', 'layer_out/25', 'layer_out/26', 'layer_out/27', 'layer_out/28', 'layer_out/29', 'layer_out/30', 'layer_out/31', 'layer_out/32', 'layer_out/33', 'layer_out/34', 'layer_out/35']
Loading layer='layer_out/21', cached position=1.


Scanning chunk metadata:   0%|          | 0/23 [00:00<?, ?it/s]

Reading selected layer:   0%|          | 0/4 [00:00<?, ?it/s]

Activation matrix shape: (4136, 2560)
Loaded 4,136 of 4,136 selected samples.


## 6. Prepare metadata and project with PCA

Before aggregation, the notebook normalizes each duration to `time_horizon_months` so equivalent values such as 60 minutes and 1 hour share a group. `AGG_BY` averages activation vectors for rows sharing the configured fields; use `None` or an empty list to project individual activation rows. Unless `PCA_MODEL_LOAD_PATH` is set, the final three-component projection fits scikit-learn's randomized PCA solver with a fixed seed. A loaded model only performs `transform` on the prepared matrix.

In [16]:
def flatten_scalar_metadata(metadata, prefix=''):
    flattened = {}
    for key, value in metadata.items():
        path = f'{prefix}.{key}' if prefix else key
        if isinstance(value, dict):
            flattened.update(flatten_scalar_metadata(value, path))
        elif not isinstance(value, (list, tuple, set)):
            flattened[path] = value
    return flattened


metadata_df = pd.DataFrame([flatten_scalar_metadata(selected_metadata[index]) for index in loaded_indices])
metadata_df.insert(0, 'sample_index', loaded_indices)
metadata_df.insert(1, 'absolute_token_position', absolute_positions)
unit_to_months = {
    'second': 1 / (30.4375 * 86400), 'minute': 1 / (30.4375 * 1440),
    'hour': 1 / (30.4375 * 24), 'day': 1 / 30.4375, 'week': 7 / 30.4375,
    'month': 1, 'year': 12, 'decade': 120, 'century': 1200, 'millennium': 12000,
}
unit_to_months.update({f'{unit}s': value for unit, value in list(unit_to_months.items())})
unit_to_months['centuries'] = 1200
unit_to_months['millennia'] = 12000
value_field = 'base_value' if 'base_value' in metadata_df else 'value'
unit_field = 'base_unit' if 'base_unit' in metadata_df else 'unit'
unknown_units = sorted(set(metadata_df[unit_field].astype(str).str.lower()) - set(unit_to_months))
if unknown_units:
    raise ValueError(f'Cannot convert time-horizon units to months: {unknown_units}')
metadata_df['time_horizon_months'] = [
    round(float(value) * unit_to_months[str(unit).lower()], 12)
    for value, unit in zip(metadata_df[value_field], metadata_df[unit_field])
]
available_phrasing_fields = [field for field in PHRASING_FIELDS if field in metadata_df]
aggregation_fields = list(AGG_BY or [])
missing_aggregation_fields = set(aggregation_fields) - set(metadata_df.columns)
if missing_aggregation_fields:
    raise ValueError(f'Missing AGG_BY fields: {sorted(missing_aggregation_fields)}')
if len(set(aggregation_fields)) != len(aggregation_fields):
    raise ValueError('AGG_BY must not contain duplicate fields.')

if aggregation_fields:
    vectors, rows = [], []
    groups = metadata_df.groupby(aggregation_fields, dropna=False, sort=False).indices
    for row_offsets in groups.values():
        row_offsets = list(row_offsets)
        vectors.append(activation_matrix.index_select(0, torch.as_tensor(row_offsets)).mean(dim=0))
        row = metadata_df.iloc[row_offsets[0]].copy()
        row['source_sample_count'] = len(row_offsets)
        for field in [value_field, unit_field, *available_phrasing_fields]:
            if metadata_df.iloc[row_offsets][field].nunique(dropna=True) > 1:
                row[field] = '<averaged>'
        rows.append(row)
    pca_matrix = torch.stack(vectors)
    analysis_metadata_df = pd.DataFrame(rows).reset_index(drop=True)
else:
    pca_matrix = activation_matrix
    analysis_metadata_df = metadata_df.copy()
    analysis_metadata_df['source_sample_count'] = 1

pca_input = pca_matrix.cpu().numpy()
if PCA_MODEL_LOAD_PATH is None:
    if len(pca_input) < 3:
        raise ValueError('At least three analysis rows are required to fit a 3D PCA projection.')
    pca = PCA(n_components=3, svd_solver='randomized', random_state=0)
    projections = pca.fit_transform(pca_input)
    print('Fitted PCA model on the prepared activation matrix.')
else:
    pca_model_load_path = Path(PCA_MODEL_LOAD_PATH)
    with pca_model_load_path.open('rb') as model_file:
        pca = pickle.load(model_file)
    if not isinstance(pca, PCA):
        raise TypeError(f'{pca_model_load_path} does not contain a scikit-learn PCA model.')
    projections = pca.transform(pca_input)
    print(f'Loaded PCA model from {pca_model_load_path}.')

if projections.shape[1] != 3:
    raise ValueError(f'PCA model must produce exactly 3 components, got {projections.shape[1]}.')
if PCA_MODEL_SAVE_PATH is not None:
    pca_model_save_path = Path(PCA_MODEL_SAVE_PATH)
    pca_model_save_path.parent.mkdir(parents=True, exist_ok=True)
    with pca_model_save_path.open('wb') as model_file:
        pickle.dump(pca, model_file)
    print(f'Saved PCA model to {pca_model_save_path}.')

explained_fraction = pca.explained_variance_ratio_
print('PCA input shape:', tuple(pca_matrix.shape))
print('Explained variance fractions:', explained_fraction)

Fitted PCA model on the prepared activation matrix.
PCA input shape: (94, 2560)
Explained variance fractions: [0.6738006  0.19761644 0.05270294]


## 7. Visualize the projection

The controls discover scalar prompt-metadata fields automatically. The 3D projection and paired 2D projections have independent color and filtering controls.

In [17]:
df_projs = analysis_metadata_df.copy().reset_index(drop=True)
df_projs[['PC1', 'PC2', 'PC3']] = projections
df_projs['log10_time_horizon_months'] = np.log10(df_projs['time_horizon_months'])
metadata_fields = sorted(column for column in df_projs if column not in {'PC1', 'PC2', 'PC3', 'sample_index'})
color_fields = ['log10_time_horizon_months', *[field for field in metadata_fields if field != 'log10_time_horizon_months']]
print(f'Prepared {len(df_projs):,} projected points.')

Prepared 94 projected points.


In [18]:
import ipywidgets as widgets
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from pandas.api.types import is_bool_dtype, is_numeric_dtype

try:
    from google.colab import output as colab_output
except ImportError:
    in_colab = False
else:
    in_colab = True
    colab_output.enable_custom_widget_manager()
    pio.renderers.default = 'colab'

color_dropdown = widgets.Dropdown(options=color_fields, value='log10_time_horizon_months', description='Color by:', layout=widgets.Layout(width='500px'))
filter_dropdown = widgets.Dropdown(options=[('(no filter)', None), *[(field, field) for field in metadata_fields]], description='Filter by:', layout=widgets.Layout(width='500px'))
filter_values = widgets.SelectMultiple(description='Keep:', rows=6, layout=widgets.Layout(width='500px'))
plot_output = widgets.Output()

def update_filter_values():
    field = filter_dropdown.value
    values = [] if field is None else sorted(df_projs[field].dropna().unique().tolist(), key=str)
    filter_values.options = [(str(value), value) for value in values]
    filter_values.value = tuple(values)
    filter_values.disabled = field is None

def render_plot(change=None):
    filtered = df_projs
    if filter_dropdown.value is not None:
        filtered = filtered[filtered[filter_dropdown.value].isin(filter_values.value)]
    plot_output.clear_output(wait=True)
    with plot_output:
        if filtered.empty:
            print('No points match the selected filter values.')
            return
        color_field = color_dropdown.value
        plot_data = filtered.copy()
        numeric_color = is_numeric_dtype(plot_data[color_field]) and not is_bool_dtype(plot_data[color_field])
        if not numeric_color:
            plot_data[color_field] = plot_data[color_field].astype('string').fillna('<missing>')
        title = f'{selected_layer}, cached position {POSITION_INDEX} ({len(plot_data):,} points)'
        hover_fields = ['sample_index', 'time_horizon_months']
        fig_3d = px.scatter_3d(plot_data, x='PC1', y='PC2', z='PC3', color=color_field, color_continuous_scale='Viridis' if numeric_color else None, hover_data=hover_fields, title=title, opacity=0.7)
        fig_3d.update_traces(marker={'size': 4})

        if in_colab:
            fig_3d.show(renderer='colab')
        else:
            display(fig_3d)

def on_filter_change(change):
    update_filter_values()
    render_plot()

color_dropdown.observe(render_plot, names='value')
filter_dropdown.observe(on_filter_change, names='value')
filter_values.observe(render_plot, names='value')
update_filter_values()
display(widgets.VBox([color_dropdown, filter_dropdown, filter_values, plot_output]))
render_plot()

### Paired 2D projections

These controls independently configure the PC1–PC2 and PC1–PC3 views.

In [19]:
color_dropdown_2d = widgets.Dropdown(options=color_fields, value='log10_time_horizon_months', description='2D color:', layout=widgets.Layout(width='500px'))
filter_dropdown_2d = widgets.Dropdown(options=[('(no filter)', None), *[(field, field) for field in metadata_fields]], description='2D filter:', layout=widgets.Layout(width='500px'))
filter_values_2d = widgets.SelectMultiple(description='2D keep:', rows=6, layout=widgets.Layout(width='500px'))
plot_output_2d = widgets.Output()

def update_filter_values_2d():
    field = filter_dropdown_2d.value
    values = [] if field is None else sorted(df_projs[field].dropna().unique().tolist(), key=str)
    filter_values_2d.options = [(str(value), value) for value in values]
    filter_values_2d.value = tuple(values)
    filter_values_2d.disabled = field is None

def render_plot_2d(change=None):
    filtered = df_projs
    if filter_dropdown_2d.value is not None:
        filtered = filtered[filtered[filter_dropdown_2d.value].isin(filter_values_2d.value)]
    plot_output_2d.clear_output(wait=True)
    with plot_output_2d:
        if filtered.empty:
            print('No points match the selected 2D filter values.')
            return
        color_field = color_dropdown_2d.value
        plot_data = filtered.copy()
        numeric_color = is_numeric_dtype(plot_data[color_field]) and not is_bool_dtype(plot_data[color_field])
        if not numeric_color:
            plot_data[color_field] = plot_data[color_field].astype('string').fillna('<missing>')
        projection_pairs = pd.concat([
            plot_data.assign(_projection_pair='PC1 vs PC2', _pc_x=plot_data['PC1'], _pc_y=plot_data['PC2']),
            plot_data.assign(_projection_pair='PC1 vs PC3', _pc_x=plot_data['PC1'], _pc_y=plot_data['PC3']),
        ], ignore_index=True)
        title = f'{selected_layer}, cached position {POSITION_INDEX} ({len(plot_data):,} points)'
        fig_2d = px.scatter(
            projection_pairs,
            x='_pc_x',
            y='_pc_y',
            color=color_field,
            facet_col='_projection_pair',
            category_orders={'_projection_pair': ['PC1 vs PC2', 'PC1 vs PC3']},
            color_continuous_scale='Viridis' if numeric_color else None,
            hover_data=['sample_index', 'time_horizon_months'],
            labels={'_pc_x': 'PC1', '_pc_y': 'Component value', '_projection_pair': ''},
            title=f'2D projections - {title}',
            opacity=0.7,
        )
        fig_2d.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))
        fig_2d.update_traces(marker={'size': 5})
        fig_2d.update_xaxes(title_text='PC1')
        fig_2d.update_yaxes(title_text='PC2', col=1)
        fig_2d.update_yaxes(title_text='PC3', col=2)
        if in_colab:
            fig_2d.show(renderer='colab')
        else:
            display(fig_2d)

def on_filter_change_2d(change):
    update_filter_values_2d()
    render_plot_2d()

color_dropdown_2d.observe(render_plot_2d, names='value')
filter_dropdown_2d.observe(on_filter_change_2d, names='value')
filter_values_2d.observe(render_plot_2d, names='value')
update_filter_values_2d()
display(widgets.VBox([color_dropdown_2d, filter_dropdown_2d, filter_values_2d, plot_output_2d]))
render_plot_2d()

In [20]:
df_projs

,sample_index,absolute_token_position,template_id,template_metadata.prompt_framing,template_metadata.output_format,task,task_metadata.task_family,task_metadata.difficulty,task_metadata.domain,task_metadata.complexity,...,number_format,value,value_text,unit,time_horizon_months,source_sample_count,PC1,PC2,PC3,log10_time_horizon_months
0,8272,49,task_available_time__approach_actions,task_available_time,approach_actions,answer a yes-or-no question,quick_decision,low,communication,low,...,numeric,1,1,second,3.802570e-07,8,-11.890897,6.825058,0.366723,-6.419923
1,8274,49,task_available_time__approach_actions,task_available_time,approach_actions,answer a yes-or-no question,quick_decision,low,communication,low,...,numeric,2,2,seconds,7.605140e-07,8,-11.870760,6.367583,0.649121,-6.118893
2,8276,49,task_available_time__approach_actions,task_available_time,approach_actions,answer a yes-or-no question,quick_decision,low,communication,low,...,numeric,3,3,seconds,1.140771e-06,8,-11.737741,6.222656,0.810011,-5.942802
3,8278,49,task_available_time__approach_actions,task_available_time,approach_actions,answer a yes-or-no question,quick_decision,low,communication,low,...,numeric,4,4,seconds,1.521028e-06,8,-11.455744,5.580929,0.740287,-5.817863
4,8280,49,task_available_time__approach_actions,task_available_time,approach_actions,answer a yes-or-no question,quick_decision,low,communication,low,...,numeric,5,5,seconds,1.901285e-06,8,-11.599728,5.992931,0.844043,-5.720953
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,9990,53,task_available_time__approach_actions,task_available_time,approach_actions,develop a long-term strategy for humanity's su...,civilization_survival,very_high,civilization,very_high,...,numeric,20,20,millennia,2.400000e+05,8,9.273323,4.307544,0.316885,5.380211
90,9994,53,task_available_time__approach_actions,task_available_time,approach_actions,develop a long-term strategy for humanity's su...,civilization_survival,very_high,civilization,very_high,...,numeric,30,30,millennia,3.600000e+05,8,9.352588,4.559090,0.106102,5.556303
91,9998,53,task_available_time__approach_actions,task_available_time,approach_actions,develop a long-term strategy for humanity's su...,civilization_survival,very_high,civilization,very_high,...,numeric,50,50,millennia,6.000000e+05,8,9.234695,4.388433,0.249782,5.778151
92,10002,53,task_available_time__approach_actions,task_available_time,approach_actions,develop a long-term strategy for humanity's su...,civilization_survival,very_high,civilization,very_high,...,numeric,70,70,millennia,8.400000e+05,8,9.289318,4.367348,0.027073,5.924279
